# Test month

This notebook scores the selected model on the held-out test month, **2026-04**. We do this one time only. All decisions (features, model, settings, target, decision metric) were made before, on the validation months 2025-10 to 2026-03 (`07_final_model_selection.ipynb`). Thus, nothing in this notebook can change a decision.

- **Model:** `serving.final_model()`: XGBoost (06 settings) on Dataset A + school districts + HOA + flooring + comparable sales, fit on log price.
- **Training data:** the 18 months before the test month (2024-10 to 2026-03), the same setup as each validation month.
- **Test data:** the sales that closed in 2026-04. As in Dataset A, the price limits (0.5% and 99.5%) come from the training window.
- **Price ranges:** the 80%, 90% and 95% ranges come from the validation errors only. The test month shows if they hold on new data.

In [1]:
import numpy as np
import pandas as pd

from idx_exchange_data_science.cleaning import TARGET
from idx_exchange_data_science.features import build_table
from idx_exchange_data_science.modeling import fit_window, score
from idx_exchange_data_science.paths import DATA_DIR, RESULTS_DIR
from idx_exchange_data_science.serving import RANGE_LEVELS, final_model
from idx_exchange_data_science.splits import split

pd.set_option("display.float_format", "{:,.4f}".format)

TEST_MONTH = "2026-04"
X = 18
MODEL = "XGBoost (06 settings) | Districts + HOA + flooring + comps | log price"  # its name in 07

df = build_table()
model = fit_window(final_model(), df, TEST_MONTH, X)
train, test = split(df, TEST_MONTH, X)
estimate = pd.Series(model.predict(test), index=test.index)
print(f"trained on {len(train):,} sales ({train['CloseDate'].min():%Y-%m} to {train['CloseDate'].max():%Y-%m}), "
      f"tested on {len(test):,} sales in {TEST_MONTH}")

trained on 189,534 sales (2024-10 to 2026-03), tested on 12,309 sales in 2026-04


## 1. Test scores
The validation column is the mean over the 6 validation months, from `07`.

In [2]:
validation = pd.read_csv(RESULTS_DIR / "final_model_selection.csv").set_index("model").loc[MODEL]
metrics = ["median_ape", "r2", "mae", "mape", "within_10pct"]
test_scores = pd.Series(score(test[TARGET], estimate))[metrics]
pd.DataFrame({"validation (2025-10 to 2026-03)": validation[metrics], f"test ({TEST_MONTH})": test_scores})

,validation (2025-10 to 2026-03),test (2026-04)
median_ape,0.0723,0.0740
r2,0.9132,0.9152
mae,"136,021.7242","143,886.7983"
mape,0.1062,0.1056
within_10pct,0.6272,0.6209


## 2. Price ranges
Each range is the estimate times the central 80%, 90% or 95% of the validation ratios (sale price / estimate). The share of test homes inside each range should be close to the level.

In [3]:
validation_estimates = pd.read_parquet(DATA_DIR / "experiments" / "final_model_selection_predictions.parquet")[MODEL]
ratios = df.loc[validation_estimates.index, TARGET] / validation_estimates
coverage = {}
for level in RANGE_LEVELS:
    low, high = ratios.quantile([(1 - level) / 2, (1 + level) / 2])
    inside = test[TARGET].between(estimate * low, estimate * high)
    coverage[f"{level:.0%} range"] = {"ratio limits": f"{low:.3f} to {high:.3f}", "test homes inside": inside.mean()}
pd.DataFrame(coverage).T

,ratio limits,test homes inside
80% range,0.838 to 1.153,0.7812
90% range,0.777 to 1.226,0.8912
95% range,0.720 to 1.311,0.9456


## 3. Test error by distance to the comparable sales
The worst-error review (`experiments/07b_worst_errors.ipynb`) found that the error increases with this distance on the validation months. Does the test month show the same?

In [4]:
error = (estimate / test[TARGET] - 1).abs()
distance = pd.cut(test["CompDistanceKm"], [0, 0.5, 1, 2, 5, np.inf], labels=["under 0.5 km", "0.5 to 1 km", "1 to 2 km", "2 to 5 km", "over 5 km"])
error.groupby(distance, observed=True).agg(homes="size", median="median", p90=lambda e: e.quantile(0.9))

,homes,median,p90
CompDistanceKm,,,
under 0.5 km,9763,0.0700,0.2111
0.5 to 1 km,1735,0.0796,0.2485
1 to 2 km,427,0.1090,0.2980
2 to 5 km,203,0.1196,0.4142
over 5 km,96,0.1584,0.4280


**Observations**
- **The test scores are close to the validation scores.** MdAPE 7.40% (validation 7.23%), R² 0.9152 (0.9132), 62.1% of homes within 10% (62.7%). Thus, the validation months did not give an optimistic picture.
- **MAE is higher** (\$143.9K against \$136.0K). Prices in the test month are higher, so the dollar errors are higher too. The percentage errors are almost the same.
- **The price ranges hold, but they are slightly narrow.** 78.1% of the test homes are inside the 80% range, 89.1% inside the 90% range and 94.6% inside the 95% range.
- **The error increases with the distance to the comparable sales, as in validation:** median 7.0% within 0.5 km, 15.8% at more than 5 km. This supports price ranges by distance group.
- **Final result:** MdAPE 7.40%, R² 0.915 on the test month (12,309 sales). We do not change the model because of these scores.

In [5]:
pd.DataFrame({"metric": metrics, "validation": validation[metrics].to_numpy(), "test": test_scores.to_numpy()}).assign(
    model=MODEL, test_month=TEST_MONTH, n_test=len(test)).to_csv(RESULTS_DIR / "test_month.csv", index=False)